# From Recipe Annotation to Knowledge Graph
## CMC Process Ontology — Hands-On Demonstration

This notebook shows how a **process description annotated with CMC-PO terms**
can be converted into a **machine-readable RDF knowledge graph** and visualized.

**What you will see:**
1. Load the annotation CSV (the exercise you completed on paper during the workshop)
2. Build an RDF graph — each recipe step becomes a connected node
3. Visualize the recipe as an interactive network diagram

> **No Python knowledge required.** Just follow along as we run each cell.

## Step 0 — Install Required Libraries
Run this cell once to install the packages we need.

In [ ]:
# Install required packages (run once)
!pip install rdflib pandas networkx matplotlib pyvis -q

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, RDFS, XSD, OWL
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyvis.network import Network
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

print("All libraries loaded successfully!")

## Step 2 — Load the Annotation CSV

This CSV contains the **Paracetamol API Synthesis process description annotated with CMC-PO ontology terms**.
Each row maps a line from the recipe to its corresponding:
- **Process Stage** (e.g., Synthesis)
- **Process Operation** (e.g., Prepare, Fill, React)
- **Process Action** (e.g., Charge, Set, Hold, Confirm)

This is exactly the same annotation you did on paper during the exercise!

In [ ]:
# Load the annotation spreadsheet
df = pd.read_csv("Process_Description_Annotation_Excercise.csv")

# Clean up number columns (remove decimal points from IDs)
for col in ["Stage Number", "Operation Number", "Action Number"]:
    df[col] = df[col].apply(
        lambda x: str(int(float(x))) if pd.notna(x) and str(x) not in ("", "nan")
        and str(x).replace(".", "").replace("-", "").isdigit() else ""
    )

# Show key columns for a quick overview
display_cols = [
    "Corresponding line from the recipe",
    "Process Stage Ontology Term Label",
    "Process Operation Ontology Term",
    "Process Action Ontology Term"
]

print(f"Loaded {len(df)} annotated recipe lines\n")
df[display_cols].head(15)

## Step 3 — Build the RDF Knowledge Graph

Now we transform the flat CSV table into a **connected graph**.

Each recipe instruction becomes a **node**, and the relationships between
Process → Stage → Operation → Action become **edges** (connections).


In [ ]:
# ── Define namespaces (like address prefixes for our ontology terms) ──
CMCPO = Namespace("https://spec.pistoiaalliance.org/cmc/ontology/")
IOF   = Namespace("https://spec.industrialontologies.org/ontology/core/Core/")
IOFC  =  Namespace("https://spec.industrialontologies.org/ontology/construct/")
EX    = Namespace("http://example.org/paracetamol-recipe/")
OBO   = Namespace("http://purl.obolibrary.org/obo/")


# Create an empty RDF graph
g = Graph()
g.bind("cmcpo",   CMCPO)
g.bind("iof",   IOF)
g.bind("iofc",   IOFC)
g.bind("ex",    EX)
g.bind("rdfs",    RDFS)
g.bind("obo",    OBO)

# ── Create the top-level Process node ──
process_uri = EX["ParacetamolAPISynthesis"]
g.add((process_uri, RDF.type, IOF["PlanSpecification"]))
g.add((process_uri, RDFS.label, Literal("Paracetamol API Synthesis Process Description")))

# ── Track nodes we've already created (avoid duplicates) ──
created_stages = {}
created_operations = {}
created_actions = {}

# ── Helper: make a clean URI-safe name ──
def safe_uri(text):
    return text.strip().replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")

# ── Walk through each annotated row and build the graph ──
for _, row in df.iterrows():
    stage_uri = None
    op_uri = None
    act_uri = None

    recipe_line = str(row.get("Corresponding line from the recipe", "")).strip()
    if not recipe_line:
        continue

    # --- Stage ---
    stage_label = str(row.get("Process Stage Ontology Term Label", "")).strip()
    stage_num   = str(row.get("Stage Number", "")).strip()
    stage_iri   = str(row.get("Process Stage Ontology Term IRI", "")).strip()
    
    stage_uri = None
    if stage_label and stage_label not in ("nan", ""):
        if stage_num and stage_num not in ("nan", ""):
            stage_key = f"{stage_num}_{stage_label}"
            if stage_key not in created_stages:
                stage_uri = EX[f"Stage_{stage_num}_{safe_uri(stage_label)}"]
                g.add((stage_uri, RDF.type, IOFC["RecipeProcessStage"]))
                g.add((stage_uri, RDFS.label, Literal(f"Stage {stage_num}: {stage_label}")))
                if stage_iri and stage_iri.startswith("http"):
                    stage_prescribed_instance_iri = EX[f"process/stage/{safe_uri(stage_label)}"]
                    g.add((stage_prescribed_instance_iri, RDF.type, URIRef(stage_iri)))
                    g.add((stage_uri, IOF["prescribes"], stage_prescribed_instance_iri))
                g.add((stage_uri, OBO["BFO_0000177"], process_uri))
                created_stages[stage_key] = stage_uri
            else:
                stage_uri = created_stages[stage_key]

    # --- Operation ---
    op_label = str(row.get("Process Operation Ontology Term", "")).strip()
    op_num   = str(row.get("Operation Number", "")).strip()
    op_iri   = str(row.get("Process Operation Ontology IRI", "")).strip()

    op_uri = None
    # Skip placeholder entries where no suitable class was found
    if (op_label and op_label not in ("nan", "")
        and "No suitable" not in op_label
        and "should be added" not in op_label
        and op_num and op_num not in ("nan", "")):

        op_key = f"{op_num}_{op_label}"
        if op_key not in created_operations:
            op_uri = EX[f"Op_{op_num}_{safe_uri(op_label)}"]
            g.add((op_uri, RDF.type, IOFC["RecipeProcessOperation"]))
            g.add((op_uri, RDFS.label, Literal(f"Op {op_num}: {op_label}")))
            if op_iri and op_iri.startswith("http"):
                op_prescribed_instance_iri = EX[f"process/operation/{safe_uri(op_label)}"]
                g.add((op_prescribed_instance_iri, RDF.type, URIRef(op_iri)))
                g.add((op_uri, IOF["prescribes"], op_prescribed_instance_iri))
            # Link operation to its parent stage
            if stage_uri:
                g.add((op_uri, OBO["BFO_0000177"], stage_uri))
            created_operations[op_key] = op_uri
        else:
            op_uri = created_operations[op_key]

    # --- Action ---
    act_label = str(row.get("Process Action Ontology Term", "")).strip()
    act_num   = str(row.get("Action Number", "")).strip()
    act_iri   = str(row.get("Process Action Ontology IRI", "")).strip()

    if (act_label and act_label not in ("nan", "")
        and "should be added" not in act_label
        and act_num and act_num not in ("nan", "")):

        act_key = f"{act_num}_{act_label}"
        if act_key not in created_actions:
            act_uri = EX[f"Act_{act_num}_{safe_uri(act_label)}"]
            g.add((act_uri, RDF.type, IOFC["RecipeProcessAction"]))
            g.add((act_uri, RDFS.label, Literal(act_label)))
            g.add((act_uri, RDFS["comment"], Literal(recipe_line)))
            if act_iri and act_iri.startswith("http"):
                action_prescribed_instance_iri = EX[f"process/action/{safe_uri(act_label)}"]
                g.add((action_prescribed_instance_iri, RDF.type, URIRef(act_iri)))
                g.add((act_uri, IOF["prescribes"], action_prescribed_instance_iri))
            # Link action to its parent operation
            if op_uri:
                g.add((act_uri, OBO["BFO_0000177"], op_uri))
            created_actions[act_key] = act_uri

# ── Summary ──
print("═" * 50)
print("  RDF KNOWLEDGE GRAPH — BUILD SUMMARY")
print("═" * 50)
print(f"  Total RDF triples : {len(g)}")
print(f"  Process Stages    : {len(created_stages)}")
for k in created_stages:
    print(f"     → {k.split('_', 1)[1]}")
print(f"  Operations        : {len(created_operations)}")
for k in created_operations:
    print(f"     → {k.split('_', 1)[1]} (Op {k.split('_', 1)[0]})")
print(f"  Actions           : {len(created_actions)}")
print("═" * 50)

In [ ]:
# Serialize the graph to Turtle format and display a preview
turtle_output = g.serialize(format="turtle")

# Save to file
with open("paracetamol_synthesis_process_desc.ttl", "w") as f:
    f.write(turtle_output)

# Show first 50 lines as a preview
lines = turtle_output.split("\n")
print(f"Turtle file saved: paracetamol_recipe.ttl ({len(lines)} lines total)")
print("═" * 70)
print("\n".join(lines[:50]))
if len(lines) > 50:
    print(f"\n... ({len(lines) - 50} more lines — see full file for details)")

In [ ]:
# Serialize the graph to Turtle format and display a preview
turtle_output = g.serialize(format="turtle")

# Save to file
with open("paracetamol_recipe.ttl", "w") as f:
    f.write(turtle_output)

# Show first 80 lines as a preview
lines = turtle_output.split("\n")
print(f"Turtle file saved: paracetamol_recipe.ttl ({len(lines)} lines total)")
print("═" * 70)
print("\n".join(lines[:80]))
if len(lines) > 80:
    print(f"\n... ({len(lines) - 80} more lines — see full file for details)")

## Step 3 — Visualize: Interactive Graph Overview

Your annotated recipe rendered as a **visual network**.

- 🔵 **Blue** = Process (the overall recipe)
- 🟢 **Green** = Stage (Synthesis, Crystallization)
- 🟠 **Orange** = Operation (Prepare, Fill, React, ...)
- 🔴 **Red** = Action (Charge, Set, Hold, Confirm, ...)

Each connection is a formal, machine-readable relationship.

- **Hover over any action node** to see the full recipe instruction it represents
- **Drag** nodes to rearrange the layout
- **Scroll** to zoom in and out

In [ ]:
# ── Build a NetworkX graph from the RDF for visualization ──
G = nx.DiGraph()  # Detailed graph for interactive view

# Build graph
G.add_node("Paracetamol\nManufacturing", level="Process")

for key, uri in created_stages.items():
    label = key.split("_", 1)[1] if "_" in key else key
    G.add_node(label, level="Stage")
    G.add_edge(label, "Paracetamol\nManufacturing", relation="obo:BFO_0000177")

for key, uri in created_operations.items():
    parts = key.split("_", 1)
    op_num = parts[0]
    op_label = parts[1] if len(parts) > 1 else key
    node_label = f"{op_label}\n(Op {op_num})"
    G.add_node(node_label, level="Operation")
    for skey in created_stages:
        stage_num = skey.split("_", 1)[0]
        if op_num.startswith(stage_num):
            stage_label = skey.split("_", 1)[1]
            G.add_edge(node_label, stage_label, relation="obo:BFO_0000177")
            break

# ── Detailed graph: individual actions with recipe instructions ──
for key, uri in created_actions.items():
    parts = key.split("_", 1)
    act_num = parts[0]
    act_label = parts[1] if len(parts) > 1 else key

    # Retrieve the recipe instruction from the RDF graph
    instruction = ""
    for s, p, o in g.triples((uri, RDFS["comment"], None)):
        instruction = str(o).strip()
        # Clean up trailing non-breaking spaces
        instruction = instruction.replace("\xa0", " ").strip()
        break

    # Build a two-line label: Action type + recipe text
    if instruction:
        short_instr = instruction[:75] + "..." if len(instruction) > 75 else instruction
        node_id = f"act_{act_num}"  # unique ID
        display_label = f"{act_label}\n» {short_instr}"
    else:
        node_id = f"act_{act_num}"
        display_label = act_label

    G.add_node(node_id, level="Action", display=display_label,
                      action_type=act_label, instruction=instruction)

    # Link to parent operation
    parent_op_node = None
    for okey in created_operations:
        o_num = okey.split("_", 1)[0]
        if act_num.startswith(o_num):
            o_label = okey.split("_", 1)[1]
            parent_op_node = f"{o_label}\n(Op {o_num})"
            break
    if parent_op_node and parent_op_node in G.nodes:
        G.add_edge(node_id, parent_op_node, relation="obo:BFO_0000177")

print(f"Detailed graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Action nodes show only the action label; hover to see the full recipe instruction

net = Network(
    height="800px", width="100%",
    bgcolor="#FAFAFA", font_color="#333333",
    directed=True, notebook=True, cdn_resources="in_line"
)

# Physics + layout settings for a clean hierarchical display
net.set_options('''
{
  "physics": {
    "hierarchicalRepulsion": {
      "centralGravity": 0.2,
      "springLength": 250,
      "nodeDistance": 200,
      "springConstant": 0.01
    },
    "solver": "hierarchicalRepulsion"
  },
  "layout": {
    "hierarchical": {
      "enabled": true,
      "direction": "UD",
      "sortMethod": "directed",
      "levelSeparation": 220,
      "nodeSpacing": 200
    }
  }
}
''')

# Visual config per PSOA level
vis_config = {
    "Process":   {"color": "#2E86C1", "size": 50, "shape": "diamond", "font_size": 20},
    "Stage":     {"color": "#27AE60", "size": 38, "shape": "dot",     "font_size": 16},
    "Operation": {"color": "#E67E22", "size": 28, "shape": "dot",     "font_size": 13},
    "Action":    {"color": "#E74C3C", "size": 18, "shape": "dot",     "font_size": 10},
}

level_map = {"Process": 0, "Stage": 1, "Operation": 2, "Action": 3}

for node in G.nodes:
    data = G.nodes[node]
    lvl = data.get("level", "Action")
    cfg = vis_config.get(lvl, vis_config["Action"])

    if lvl == "Action":
        # Action nodes: simple circle with just the action type label
        # Full recipe instruction appears only on hover
        action_type = data.get("action_type", "")
        instruction = data.get("instruction", "").replace("\xa0", " ").strip()

        tooltip = f"Action: {action_type}\n\nRecipe instruction:\n{instruction}" if instruction else f"Action: {action_type}"

        net.add_node(
            node,
            label=action_type,
            color=cfg["color"],
            size=cfg["size"],
            shape="dot",
            font={"size": cfg["font_size"], "color": "#333", "face": "arial"},
            title=tooltip,
            level=level_map["Action"],
        )

    else:
        # Process / Stage / Operation nodes
        display_label = node.replace("\n", " ")
        tooltip = f"PSOA Level: {lvl}\nLabel: {display_label}"

        net.add_node(
            node,
            label=display_label,
            color=cfg["color"],
            size=cfg["size"],
            shape=cfg["shape"],
            font={"size": cfg["font_size"], "color": "#fff", "face": "arial"},
            title=tooltip,
            level=level_map.get(lvl, 3),
        )

# ── Add edges ──
for src, dst in G.edges:
    rel = G.edges[src, dst].get("relation", "")
    edge_color = "#D5D8DC"
    net.add_edge(src, dst, title=rel, color=edge_color, width=2)

# Render inline
net.save_graph("psoa_interactive.html")

with open("psoa_interactive.html", "r") as f:
    html_content = f.read()
display(HTML(html_content))

print(f"Interactive graph: {G.number_of_nodes()} nodes — hover over any action to see the recipe instruction")
print("Saved as psoa_interactive.html")